# NSE-Alpha v1 — Phase 1 Research Notebook

**Pipeline:** Data Download → Feature Engineering → Label Generation → Walk-Forward Training → Backtest → Metrics

Run cells top to bottom. First run takes 20–40 min (data download). Subsequent runs use cache (~5 min).

In [ ]:
import sys
from pathlib import Path

repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

sns.set_theme(style='darkgrid', palette='husl')
plt.rcParams['figure.figsize'] = (14, 5)

print('Libraries loaded ✓')

## 1. Configuration

In [ ]:
from config.settings import *
from data.universe import get_universe

universe = get_universe()
print(f'Universe    : {len(universe)} tickers')
print(f'History     : {HISTORY_START} to {HISTORY_END}')
print(f'Target      : {FORWARD_DAYS}-day forward return')
print(f'Buy thresh  : >{BUY_THRESHOLD:.1%}')
print(f'Sell thresh : <{SELL_THRESHOLD:.1%}')
print(f'FRED macro  : {ENABLE_MACRO_FRED}')

## 2. Data Download

In [ ]:
from data.downloader import download_all

# force=False uses cached parquet files if < 20 hours old
datasets = download_all(force=False)

print(f"OHLCV tickers : {len(datasets['ohlcv'])}")
print(f"Index shape   : {datasets['index'].shape}")
print(f"VIX rows      : {len(datasets['vix'])}")
macro_shape = datasets['macro'].shape if not datasets['macro'].empty else 'skipped'
print(f"Macro shape   : {macro_shape}")

In [ ]:
# Sanity check on a sample ticker
sample_ticker = 'TCS'
df_sample = datasets['ohlcv'].get(sample_ticker, pd.DataFrame())

if not df_sample.empty:
    print(f'{sample_ticker}: {len(df_sample)} trading days')
    print(f'Date range  : {df_sample.index[0].date()} to {df_sample.index[-1].date()}')
    print(f'Missing AdjClose: {df_sample["Adj Close"].isna().sum()}')
    display(df_sample.tail(5))

In [ ]:
# Nifty 50 index chart
nifty = datasets['index']['nifty50'].dropna()
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(nifty.index, nifty.values, color='#26a641', linewidth=1.5)
ax.set_title('Nifty 50 — 2014 to 2024', fontsize=13)
ax.set_ylabel('Index Level')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
from features.pipeline import build_feature_panel, load_feature_panel

panel_path = OUTPUTS_DIR.parent / 'feature_panel.parquet'

if panel_path.exists():
    print('Loading cached feature panel...')
    panel = load_feature_panel()
else:
    print('Building feature panel — this takes a few minutes...')
    panel = build_feature_panel(datasets, save=True)

print(f'Panel shape  : {panel.shape}')
print(f'Tickers      : {panel["ticker"].nunique()}')
print(f'Date range   : {panel.index.min().date()} to {panel.index.max().date()}')
print(f'Feature cols : {panel.shape[1] - 3}')

In [ ]:
# Label distribution
label_map = {0: 'Sell', 1: 'Hold', 2: 'Buy'}
dist = panel['signal'].value_counts().sort_index()
dist.index = [label_map[i] for i in dist.index]

fig, ax = plt.subplots(figsize=(6, 4))
colors_map = {'Sell': '#da3633', 'Hold': '#6e7681', 'Buy': '#26a641'}
bars = ax.bar(dist.index, dist.values, color=[colors_map[l] for l in dist.index])
for bar, val in zip(bars, dist.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{val:,}\n({val/len(panel):.1%})', ha='center', va='bottom', fontsize=10)
ax.set_title('Signal Label Distribution', fontsize=12)
ax.set_ylabel('Count')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
# Feature columns overview
from models.train import get_feature_columns
feat_cols = get_feature_columns(panel)
print(f'Total features: {len(feat_cols)}')
print()
for i, col in enumerate(feat_cols, 1):
    print(f'  {i:3d}. {col}')

## 4. Walk-Forward Model Training

In [ ]:
from models.train import run_training_pipeline

print(f'Folds         : {CV_FOLDS}')
print(f'OOS window    : {WF_WINDOW_DAYS} days per fold (~6 months)')
print(f'Model         : LightGBM multiclass (Sell=0 / Hold=1 / Buy=2)')
print()

results = run_training_pipeline(panel)
oos_df = results['oos_df']

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_true = oos_df['signal'].astype(int)
y_pred = oos_df['pred_signal'].astype(int)

cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
disp = ConfusionMatrixDisplay(cm, display_labels=['Sell', 'Hold', 'Buy'])

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Walk-Forward OOS Confusion Matrix', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
fi_path = OUTPUTS_DIR.parent / 'feature_importance.csv'
if fi_path.exists():
    fi = pd.read_csv(fi_path).head(25)
    fig, ax = plt.subplots(figsize=(10, 8))
    fi.sort_values('importance').plot(
        kind='barh', x='feature', y='importance', ax=ax,
        color='steelblue', legend=False
    )
    ax.set_title('Top 25 Feature Importances (Gain)', fontsize=12)
    ax.set_xlabel('Importance (Gain)')
    plt.tight_layout()
    plt.show()
else:
    print('Feature importance file not found — run training first')

In [ ]:
# Confidence distribution of OOS predictions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
label_names = {0: 'Sell', 1: 'Hold', 2: 'Buy'}
colors_list = ['#da3633', '#6e7681', '#26a641']

for i, (signal_val, color) in enumerate(zip([0, 1, 2], colors_list)):
    subset = oos_df[oos_df['pred_signal'] == signal_val]['confidence']
    axes[i].hist(subset, bins=30, color=color, alpha=0.8, edgecolor='white')
    axes[i].set_title(f'{label_names[signal_val]} — n={len(subset):,}', fontsize=11)
    axes[i].set_xlabel('Confidence')
    axes[i].set_ylabel('Count')
    axes[i].axvline(MIN_CONFIDENCE, color='white', linestyle='--', alpha=0.7,
                    label=f'Min threshold ({MIN_CONFIDENCE:.0%})')
    axes[i].legend(fontsize=8)

plt.suptitle('OOS Prediction Confidence by Signal Class', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Backtest

In [ ]:
from backtest.engine import run_backtest, INITIAL_CAPITAL

print(f'Initial capital  : INR {INITIAL_CAPITAL:,.0f}')
print(f'Stop-loss        : {STOP_LOSS_PCT:.1%}')
print(f'Take-profit      : {TAKE_PROFIT_PCT:.1%}')
print(f'Hold period      : {HOLD_PERIOD_DAYS} days')
print(f'Max positions    : {MAX_OPEN_POSITIONS}')
print(f'Transaction cost : {TRANSACTION_COST:.2%} + {SLIPPAGE:.2%} slippage')
print()

equity_df, trade_log = run_backtest(oos_df, datasets['ohlcv'])

# Save for dashboard
equity_df.to_parquet(OUTPUTS_DIR.parent / 'equity_curve.parquet')
if not trade_log.empty:
    trade_log.to_parquet(OUTPUTS_DIR.parent / 'trade_log.parquet')

print(f'\nBacktest complete')
print(f'Total trades     : {len(trade_log)}')
print(f'Date range       : {equity_df.index[0].date()} to {equity_df.index[-1].date()}')

In [ ]:
# Equity curve vs Nifty 50 benchmark
nifty_aligned = datasets['index']['nifty50'].reindex(equity_df.index).ffill()
nifty_scaled  = nifty_aligned / nifty_aligned.iloc[0] * INITIAL_CAPITAL

fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]})

# Equity curve
axes[0].plot(equity_df.index, equity_df['value'], color='#26a641',
             linewidth=2, label='NSE-Alpha v1')
axes[0].plot(nifty_scaled.index, nifty_scaled.values, color='#6e7681',
             linewidth=1.5, linestyle='--', label='Nifty 50 (buy & hold)', alpha=0.8)
axes[0].set_title('Portfolio Equity Curve vs Nifty 50 Benchmark', fontsize=13)
axes[0].set_ylabel('Portfolio Value (INR)')
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'₹{x/1e5:.1f}L'))
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Drawdown
axes[1].fill_between(equity_df.index, equity_df['drawdown'] * 100, 0,
                     color='#da3633', alpha=0.6, label='Drawdown')
axes[1].axhline(MAX_DRAWDOWN_PCT * 100, color='orange', linestyle='--',
                linewidth=1, label=f'Max allowed ({MAX_DRAWDOWN_PCT:.0%})')
axes[1].set_ylabel('Drawdown %')
axes[1].set_xlabel('Date')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Performance Metrics

In [ ]:
from backtest.metrics import compute_all_metrics, print_metrics

nifty_bm = datasets['index']['nifty50'].reindex(equity_df.index).ffill()

metrics = compute_all_metrics(
    equity_df   = equity_df,
    trade_log   = trade_log,
    benchmark   = nifty_bm,
    initial_cap = INITIAL_CAPITAL,
)

print_metrics(metrics)

In [ ]:
# Trade analysis
if not trade_log.empty:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # Return distribution
    axes[0].hist(trade_log['return_pct'] * 100, bins=40,
                 color='steelblue', edgecolor='white', alpha=0.8)
    axes[0].axvline(0, color='white', linewidth=1.5)
    axes[0].set_title('Trade Return Distribution', fontsize=11)
    axes[0].set_xlabel('Return (%)')

    # Exit reason
    exit_counts = trade_log['exit_reason'].value_counts()
    axes[1].bar(exit_counts.index, exit_counts.values,
                color=['#da3633', '#26a641', '#6e7681'])
    axes[1].set_title('Exit Reasons', fontsize=11)
    axes[1].set_ylabel('Count')

    # Sector breakdown of trades
    sector_pnl = trade_log.groupby('sector')['pnl'].sum().sort_values()
    colors_sec  = ['#26a641' if v > 0 else '#da3633' for v in sector_pnl.values]
    sector_pnl.plot(kind='barh', ax=axes[2], color=colors_sec)
    axes[2].set_title('P&L by Sector', fontsize=11)
    axes[2].set_xlabel('Total P&L (INR)')
    axes[2].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))

    plt.tight_layout()
    plt.show()
else:
    print('No trades generated — check signal thresholds and confidence filter')

## 7. Sample Signals (Latest Date)

In [ ]:
from models.predict import predict_latest

signals = predict_latest(panel)

print(f"Signal date   : {signals['signal_date'].iloc[0]}")
print(f"Total tickers : {len(signals)}")
print()

actionable = signals[signals['actionable']]
print(f"Actionable signals (conf >= {MIN_CONFIDENCE:.0%}): {len(actionable)}")
print()

buy_signals = actionable[actionable['signal'] == 2][[
    'ticker', 'signal_label', 'confidence',
    'expected_5d_return', 'entry_price'
]].head(10)

print('Top Buy Signals:')
display(buy_signals.style.format({
    'confidence': '{:.1%}',
    'expected_5d_return': '{:+.2%}',
    'entry_price': '₹{:,.2f}'
}))

## 8. Phase 1 Complete

| Step | Status |
|---|---|
| Data download | ✅ 10yr OHLCV + macro + VIX |
| Feature engineering | ✅ 50+ features per ticker |
| Label generation | ✅ 5-day forward return |
| Walk-forward training | ✅ 5-fold LightGBM |
| Backtest | ✅ Trade simulation with all risk rules |
| Final model saved | ✅ models/saved/lgbm_final.txt |

**Next steps:**
- If metrics hit targets → move to Phase 2 (`scripts/run_daily.py`)
- If Sharpe < 1.5 or win rate < 52% → tune `LGBM_PARAMS` in `config/settings.py` or adjust thresholds
- Phase 3 dashboard: `streamlit run dashboard/app.py`